# Empirical Properties of Dependency Trees in Natural Languages
### Complete Implementation — Real Universal Dependencies Data
**Author:** Raunit Kumar

---
| Language | Treebank | Sentences |
|----------|----------|-----------|
| English | UD_English-EWT | 12,544 |
| Hindi | UD_Hindi-HDTB | 13,306 |
| German | UD_German-GSD | 13,813 |
| Japanese | UD_Japanese-GSD | 7,050 |
| Tamil | UD_Tamil-TTB | 400 |
| Sanskrit | UD_Sanskrit-Vedic | 21,477 |

**Research Objective:** Test whether natural-language dependency trees are (H1) shallower and (H2) have higher max node arity than uniformly random trees of equal size.

**How to run:** Place the six `.conllu` files in the same folder as this notebook, then run all cells top to bottom.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS & SETUP   
# ═══════════════════════════════════════════════════════════════
import os, sys, random, math, json, warnings
import networkx as nx 
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')         
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


CONLLU_FILES = {
    "English":  "en_ewt-ud-train.conllu",
    "Hindi":    "hi_hdtb-ud-train.conllu",
    "German":   "de_gsd-ud-train.conllu",
    "Japanese": "ja_gsd-ud-train.conllu",
    "Tamil":    "ta_ttb-ud-train.conllu",
    "Sanskrit": "sa_vedic-ud-train.conllu",
}

# Language colours for plots
LANG_COLORS = {
    "English":  "#4e9af1",
    "Hindi":    "#e94560",
    "German":   "#f4a261",
    "Japanese": "#06d6a0",
    "Tamil":    "#c77dff",
    "Sanskrit": "#ffd60a",
}

LANGUAGES = list(CONLLU_FILES.keys())

print("Libraries loaded successfully.")
print(f"  networkx : {nx.__version__}")
print(f"  numpy    : {np.__version__}")
import scipy; print(f"  scipy    : {scipy.__version__}")
print(f"\nLanguages: {LANGUAGES}")


Libraries loaded successfully.
  networkx : 3.5
  numpy    : 2.3.2
  scipy    : 1.16.3

Languages: ['English', 'Hindi', 'German', 'Japanese', 'Tamil', 'Sanskrit']


---
## Phase 3 — Data Preprocessing

### 3.1 CoNLL-U Format

Universal Dependencies treebanks are stored in **CoNLL-U** format. Each sentence is a block of tab-separated lines (one per token), separated by a blank line. The columns are:

```
ID  FORM  LEMMA  UPOS  XPOS  FEATS  HEAD  DEPREL  DEPS  MISC
1   The   the    DET   DT    …      4     det     …     …
2   cat   cat    NOUN  NN    …      4     nsubj   …     …
```

- **Column 7 (`HEAD`):** index of the syntactic head (0 = root of sentence)
- We skip multiword tokens (`1-2`) and empty nodes (`1.1`)
- Each sentence → one `networkx.DiGraph` with edges directed **head → dependent**

### 3.2 Quality Filters
- Sentence length: **3 ≤ n ≤ 80** tokens
- Discard sentences where the parser left cycles (should not happen in UD but checked)


In [14]:

def parse_conllu(filepath, min_len=3, max_len=80):
    """
    Parse a CoNLL-U file into a list of networkx DiGraph trees.

    Each graph:
      - Nodes : integer word indices (1-based)
      - Edges : directed edge head -> dependent

    Parameters
    ----------
    filepath : str   path to .conllu file
    min_len  : int   minimum tokens per sentence (default 3)
    max_len  : int   maximum tokens per sentence (default 80)

    Returns
    -------
    list of nx.DiGraph
    """
    trees = []
    with open(filepath, encoding='utf-8') as f:
        raw = f.read()

    for block in raw.strip().split('\n\n'):
        block = block.strip()
        if not block:
            continue

        G = nx.DiGraph()
        valid = True

        for line in block.split('\n'):
            if line.startswith('#'):         
                continue
            parts = line.split('\t')
            if len(parts) < 8:
                continue

            tok_id = parts[0]
            # skip multiword tokens (e.g. "1-2") and empty nodes (e.g. "1.1")
            if '-' in tok_id or '.' in tok_id:
                continue

            try:
                tid  = int(tok_id)
                head = int(parts[6])         
            except ValueError:
                valid = False
                break

            G.add_node(tid)
            if head != 0:                     # 0 = root, no parent edge
                G.add_edge(head, tid)

        n = len(G.nodes())
        if not valid or n < min_len or n > max_len:
            continue
        # Verify it is actually a DAG 
        if not nx.is_directed_acyclic_graph(G):
            continue

        trees.append(G)

    return trees


# ── Load all treebanks ───────────────────────────────────────────
print("Loading real Universal Dependencies treebanks...\n")
print(f"{'Language':<12} {'File':<35} {'Sentences':>10}  {'Mean tokens':>12}")
print("-" * 75)

real_data = {}
for lang, fname in CONLLU_FILES.items():
  
    for prefix in ["", "/mnt/user-data/uploads/"]:
        path = prefix + fname
        if os.path.exists(path):
            break
    else:
        print(f"  !! {fname} not found — skipping {lang}")
        continue

    trees = parse_conllu(path)
    real_data[lang] = trees
    lengths = [len(t.nodes()) for t in trees]
    print(f"{lang:<12} {fname:<35} {len(trees):>10,}  {np.mean(lengths):>11.1f}")

print(f"\nLoaded {len(real_data)} languages  |  "
      f"Total sentences: {sum(len(v) for v in real_data.values()):,}")


Loading real Universal Dependencies treebanks...

Language     File                                 Sentences   Mean tokens
---------------------------------------------------------------------------
English      en_ewt-ud-train.conllu                  11,380         17.7
Hindi        hi_hdtb-ud-train.conllu                 13,301         21.1
German       de_gsd-ud-train.conllu                  13,792         19.1
Japanese     ja_gsd-ud-train.conllu                   7,014         23.6
Tamil        ta_ttb-ud-train.conllu                     400         15.8
Sanskrit     sa_vedic-ud-train.conllu                19,736          8.0

Loaded 6 languages  |  Total sentences: 65,623


---
## Phase 4 — Random Baseline Trees (Prüfer Sequence Method)

### Why Prüfer sequences?

A **Prüfer sequence** of length *n − 2* over {1, …, *n*} encodes a unique labeled tree on *n* nodes. The bijection is exact: every sequence gives one tree, every tree has one sequence. Sampling a **uniform random** Prüfer sequence therefore gives a **uniformly random labeled tree** — the ideal null model.

### Algorithm
1. Sample sequence of length *n − 2*, each element in {1, …, *n*}
2. Decode (standard O(n) algorithm) → undirected edges
3. Pick a random root; BFS to orient all edges parent → child

For each real sentence tree with *n* nodes we generate exactly one random tree with the same *n* nodes (**1-to-1 pairing**), controlling for sentence length.


In [15]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — RANDOM TREE GENERATION (PRÜFER SEQUENCES)
# ═══════════════════════════════════════════════════════════════

def prufer_decode(sequence):
    """
    Decode a Prüfer sequence into undirected tree edges.

    Standard O(n) algorithm:
      1. Compute degree of each node from sequence
      2. Repeatedly connect smallest leaf to current sequence element
      3. Connect the two final nodes

    Parameters
    ----------
    sequence : list[int]  values in {1 .. n}

    Returns
    -------
    list of (u, v) tuples (undirected edges)
    """
    n      = len(sequence) + 2
    degree = [1] * (n + 1)        # 1-indexed; degree[0] unused

    for node in sequence:
        degree[node] += 1

    edges = []
    for node in sequence:
        for leaf in range(1, n + 1):
            if degree[leaf] == 1:
                edges.append((node, leaf))
                degree[node] -= 1
                degree[leaf] -= 1
                break

    # Connect the two remaining degree-1 nodes
    remaining = [v for v in range(1, n + 1) if degree[v] == 1]
    if len(remaining) == 2:
        edges.append((remaining[0], remaining[1]))

    return edges


def random_tree_prufer(n):
    """
    Generate a uniformly random rooted directed tree on n nodes.

    Steps:
      1. Random Prüfer sequence of length n-2
      2. Decode → undirected labeled tree
      3. Pick random root; BFS to orient edges (parent → child)

    Returns nx.DiGraph
    """
    if n == 1:
        G = nx.DiGraph(); G.add_node(1); return G
    if n == 2:
        G = nx.DiGraph(); G.add_edge(1, 2); return G

    seq      = [random.randint(1, n) for _ in range(n - 2)]
    edges    = prufer_decode(seq)
    G_undir  = nx.Graph()
    G_undir.add_nodes_from(range(1, n + 1))
    G_undir.add_edges_from(edges)

    root = random.randint(1, n)
    return nx.bfs_tree(G_undir, root)     # directed: root → children



print("=== Prüfer sequence verification ===")
for n_test in [5, 10, 20, 50]:
    T = random_tree_prufer(n_test)
    roots = [v for v, d in T.in_degree() if d == 0]
    ok = (T.number_of_nodes() == n_test and
          T.number_of_edges() == n_test - 1 and
          nx.is_directed_acyclic_graph(T) and
          len(roots) == 1)
    print(f"  n={n_test:>3}: nodes={T.number_of_nodes()}, edges={T.number_of_edges()}, "
        f"is_DAG={nx.is_directed_acyclic_graph(T)}, roots={len(roots)}  {'OK' if ok else 'Mismatch'}")

# ── Generate matched random trees ────────────────────────────────
print("\nGenerating random counterpart trees (1 per real sentence)...")
random_data = {}
for lang in real_data:
    rand_trees = [random_tree_prufer(len(t.nodes())) for t in real_data[lang]]
    random_data[lang] = rand_trees
    print(f"  {lang:<12}: {len(rand_trees):,} random trees generated")

print("\n Done. Node-count pairing check (first 3 English sentences):")
for i in range(min(3, len(real_data.get('English',[])))):
    rn = len(real_data['English'][i].nodes())
    tn = len(random_data['English'][i].nodes())
    print(f"  Sentence {i+1}: real_n={rn:>3}, rand_n={tn:>3}  {'OK' if rn==tn else 'Mismatch'}")


=== Prüfer sequence verification ===
  n=  5: nodes=5, edges=4, is_DAG=True, roots=1  OK
  n= 10: nodes=10, edges=9, is_DAG=True, roots=1  OK
  n= 20: nodes=20, edges=19, is_DAG=True, roots=1  OK
  n= 50: nodes=50, edges=49, is_DAG=True, roots=1  OK

Generating random counterpart trees (1 per real sentence)...
  English     : 11,380 random trees generated
  Hindi       : 13,301 random trees generated
  German      : 13,792 random trees generated
  Japanese    : 7,014 random trees generated
  Tamil       : 400 random trees generated
  Sanskrit    : 19,736 random trees generated

 Done. Node-count pairing check (first 3 English sentences):
  Sentence 1: real_n= 29, rand_n= 29  OK
  Sentence 2: real_n= 18, rand_n= 18  OK
  Sentence 3: real_n= 17, rand_n= 17  OK


---
## Phase 5 — Graph Metrics

### Metric Definitions

**Tree Depth**

$$\text{Depth}(T) = \max_{v \in V}\; d(\text{root},\, v)$$

The longest directed path from the root to any node (tree height). Computed via single-source BFS in O(n).

**Node Arity**

$$\text{Arity}(v) = \text{out-degree}(v)$$

$$\text{MaxArity}(T) = \max_{v \in V} \text{Arity}(v)$$

We use **MaxArity** for hypothesis H2. Mean arity = (n−1)/n is algebraically identical for every tree of n nodes (a tree always has exactly n−1 edges), so it cannot differ between real and random trees of the same size. MaxArity measures the busiest head and captures how concentrated dependency relations are.


In [16]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — METRIC COMPUTATION
# ═══════════════════════════════════════════════════════════════

def get_root(G):
    """Return the root node (in-degree 0). Returns None if absent."""
    roots = [n for n, d in G.in_degree() if d == 0]
    return roots[0] if len(roots) == 1 else None


def compute_metrics(G):
    """
    Compute structural metrics for a single dependency tree.

    Returns dict with:
      n          : number of nodes (sentence length)
      depth      : max distance root → any node  (H1 metric)
      max_arity  : max out-degree of any node     (H2 metric)
    Returns None if tree is malformed.
    """
    root = get_root(G)
    if root is None or len(G.nodes()) < 2:
        return None

    # Depth via BFS
    dist  = nx.single_source_shortest_path_length(G, root)
    depth = max(dist.values())

    # Arity = out-degree per node
    max_arity = max(d for _, d in G.out_degree())

    return {
        'n'         : len(G.nodes()),
        'depth'     : depth,
        'max_arity' : max_arity,
    }



demo = nx.DiGraph()
demo.add_nodes_from(range(1,7))
demo.add_edges_from([(3,2),(2,1),(3,4),(4,6),(6,5)])
m = compute_metrics(demo)
print("Demo: 'The cat sat on the mat'")
print(f"  Root      : {get_root(demo)}")
print(f"  Depth     : {m['depth']}  (longest path root → leaf)")
print(f"  MaxArity  : {m['max_arity']}  (node 3 has 2 children: 2 and 4)")
print()

# ── Compute for all languages ─────────────────────────────────────
print("Computing metrics for all sentences...\n")
print(f"{'Language':<12} {'N sent':>8} {'Depth_real':>11} {'Depth_rand':>11} "
      f"{'MaxAr_real':>11} {'MaxAr_rand':>11}")
print("-" * 70)

results = {}
for lang in LANGUAGES:
    if lang not in real_data:
        continue

    real_m = [compute_metrics(t) for t in real_data[lang]]
    rand_m = [compute_metrics(t) for t in random_data[lang]]

    # Keep matched pairs only
    paired = [(r, s) for r, s in zip(real_m, rand_m)
              if r is not None and s is not None]
    if not paired:
        continue
    rm, sm = zip(*paired)

    results[lang] = {
        'real_depths'    : np.array([m['depth']     for m in rm]),
        'rand_depths'    : np.array([m['depth']     for m in sm]),
        'real_max_arity' : np.array([m['max_arity'] for m in rm]),
        'rand_max_arity' : np.array([m['max_arity'] for m in sm]),
        'real_n'         : np.array([m['n']         for m in rm]),
    }

    r = results[lang]
    print(f"{lang:<12} {len(r['real_depths']):>8,} "
          f"{np.mean(r['real_depths']):>11.3f} {np.mean(r['rand_depths']):>11.3f} "
          f"{np.mean(r['real_max_arity']):>11.3f} {np.mean(r['rand_max_arity']):>11.3f}")

print("\n Metrics computed for all languages.")


Demo: 'The cat sat on the mat'
  Root      : 3
  Depth     : 3  (longest path root → leaf)
  MaxArity  : 2  (node 3 has 2 children: 2 and 4)

Computing metrics for all sentences...

Language       N sent  Depth_real  Depth_rand  MaxAr_real  MaxAr_rand
----------------------------------------------------------------------
English        11,380       3.786       7.065       4.970       2.824
Hindi          13,301       4.253       8.266       5.865       3.084
German         13,792       3.823       7.641       5.311       2.978
Japanese        7,014       4.332       8.683       5.505       3.079
Tamil             400       4.308       6.957       4.952       2.845
Sanskrit       19,736       2.974       3.934       3.023       2.168

 Metrics computed for all languages.


In [17]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — DETAILED SUMMARY STATISTICS
# ═══════════════════════════════════════════════════════════════

def summary(arr):
    return {
        'mean'  : np.mean(arr),
        'median': np.median(arr),
        'std'   : np.std(arr, ddof=1),
        'min'   : int(np.min(arr)),
        'max'   : int(np.max(arr)),
        'p25'   : np.percentile(arr, 25),
        'p75'   : np.percentile(arr, 75),
    }

for metric_key, metric_label in [
        ('real_depths',    'DEPTH'),
        ('real_max_arity', 'MAX ARITY')]:

    rand_key = metric_key.replace('real_', 'rand_')
    print("=" * 85)
    print(f"SUMMARY STATISTICS — {metric_label}")
    print("=" * 85)
    print(f"{'Language':<12} {'Cond':<8} {'Mean':>7} {'Median':>7} "
          f"{'Std':>7} {'Min':>5} {'Max':>5} {'p25':>7} {'p75':>7}")
    print("-" * 85)

    for lang in LANGUAGES:
        if lang not in results:
            continue
        for label, key in [('Real', metric_key), ('Random', rand_key)]:
            s = summary(results[lang][key])
            print(f"{lang:<12} {label:<8} "
                  f"{s['mean']:>7.2f} {s['median']:>7.2f} {s['std']:>7.2f} "
                  f"{s['min']:>5} {s['max']:>5} "
                  f"{s['p25']:>7.2f} {s['p75']:>7.2f}")
        print("-" * 85)
    print()


SUMMARY STATISTICS — DEPTH
Language     Cond        Mean  Median     Std   Min   Max     p25     p75
-------------------------------------------------------------------------------------
English      Real        3.79    4.00    1.77     1    15    2.00    5.00
English      Random      7.06    7.00    3.75     1    30    4.00    9.00
-------------------------------------------------------------------------------------
Hindi        Real        4.25    4.00    1.51     1    13    3.00    5.00
Hindi        Random      8.27    8.00    3.18     1    29    6.00   10.00
-------------------------------------------------------------------------------------
German       Real        3.82    4.00    1.41     1    12    3.00    5.00
German       Random      7.64    7.00    3.26     1    35    5.00    9.00
-------------------------------------------------------------------------------------
Japanese     Real        4.33    4.00    1.81     1    14    3.00    5.00
Japanese     Random      8.68    8.00

---
## Phase 7 — Statistical Testing

### Test choice: Mann-Whitney U

We use the **Mann-Whitney U test** (non-parametric Wilcoxon rank-sum) because:
- Depth and arity distributions are right-skewed — normality assumption of t-test fails
- Robust to outliers (unusual sentences)
- Valid for very large samples

**H1** (one-sided, `alternative='less'`): real depth < random depth  
**H2** (one-sided, `alternative='greater'`): real MaxArity > random MaxArity

### Bonferroni correction

12 tests (6 languages × 2 metrics): α′ = 0.05 / 12 ≈ 0.00417

### Effect size: Cohen's d

$$d = \frac{\bar{x}_{\text{real}} - \bar{x}_{\text{random}}}{s_{\text{pooled}}}$$

Interpretation: |d| < 0.2 negligible · 0.2–0.5 small · 0.5–0.8 medium · > 0.8 large


In [18]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — STATISTICAL TESTING
# ═══════════════════════════════════════════════════════════════

N_TESTS   = 12
ALPHA     = 0.05
ALPHA_ADJ = ALPHA / N_TESTS

def cohens_d(a, b):
    s_pool = math.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return float((np.mean(a) - np.mean(b)) / s_pool) if s_pool > 0 else 0.0

def effect_label(d):
    ad = abs(d)
    if ad < 0.2: return "negligible"
    if ad < 0.5: return "small"
    if ad < 0.8: return "medium"
    return "large"

def mwu_test(real_arr, rand_arr, alternative, alpha_adj):
    u, p = stats.mannwhitneyu(real_arr, rand_arr, alternative=alternative)
    d    = cohens_d(real_arr, rand_arr)
    sig  = (float(p) < alpha_adj) or (float(p) == 0.0)
    return dict(u=float(u), p=float(p), d=d,
                effect=effect_label(d), significant=sig,
                mean_real=float(np.mean(real_arr)),
                mean_rand=float(np.mean(rand_arr)),
                diff=float(np.mean(real_arr)-np.mean(rand_arr)))

stat_results = {}
for lang in LANGUAGES:
    if lang not in results:
        continue
    r = results[lang]
    stat_results[lang] = {
        'H1': mwu_test(r['real_depths'],    r['rand_depths'],
                        alternative='less',    alpha_adj=ALPHA_ADJ),
        'H2': mwu_test(r['real_max_arity'], r['rand_max_arity'],
                        alternative='greater', alpha_adj=ALPHA_ADJ),
    }

# ── Print results table ──────────────────────────────────────────
print(f"Bonferroni-adjusted α = {ALPHA_ADJ:.5f}  ({N_TESTS} tests)\n")
print("=" * 105)
print(f"{'Language':<12} {'Hyp':<5} {'Metric':<12} {'Mean Real':>10} {'Mean Rand':>10} "
      f"{'Diff':>9} {'p-value':>12} {'Cohen d':>9} {'Effect':>11} {'Sig?':>6}")
print("─" * 105)

h1_pass = h2_pass = 0
for lang in LANGUAGES:
    if lang not in stat_results:
        continue
    for hyp, metric in [('H1','Depth'), ('H2','MaxArity')]:
        s   = stat_results[lang][hyp]
        sig = "YES" if s['significant'] else " NO"
        p_s = f"{s['p']:.2e}" if s['p'] > 0 else "< 5e-324"
        print(f"{lang:<12} {hyp:<5} {metric:<12} "
              f"{s['mean_real']:>10.3f} {s['mean_rand']:>10.3f} "
              f"{s['diff']:>+9.3f} {p_s:>12} "
              f"{s['d']:>+9.3f} {s['effect']:>11} {sig:>6}")
        if s['significant']:
            if hyp == 'H1' and s['diff'] < 0: h1_pass += 1
            if hyp == 'H2' and s['diff'] > 0: h2_pass += 1
    print("─" * 105)

n_lang = len([l for l in LANGUAGES if l in stat_results])
print(f"\nH1 (Depth: real < random)    supported in {h1_pass}/{n_lang} languages")
print(f"H2 (MaxArity: real > random) supported in {h2_pass}/{n_lang} languages")


Bonferroni-adjusted α = 0.00417  (12 tests)

Language     Hyp   Metric        Mean Real  Mean Rand      Diff      p-value   Cohen d      Effect   Sig?
─────────────────────────────────────────────────────────────────────────────────────────────────────────
English      H1    Depth             3.786      7.065    -3.279     < 5e-324    -1.118       large    YES
English      H2    MaxArity          4.970      2.824    +2.146     < 5e-324    +1.547       large    YES
─────────────────────────────────────────────────────────────────────────────────────────────────────────
Hindi        H1    Depth             4.253      8.266    -4.014     < 5e-324    -1.611       large    YES
Hindi        H2    MaxArity          5.865      3.084    +2.781     < 5e-324    +2.489       large    YES
─────────────────────────────────────────────────────────────────────────────────────────────────────────
German       H1    Depth             3.823      7.641    -3.818     < 5e-324    -1.520       large    YES
G

---
## Phase 8 — Visualisation (5 Publication-Ready Figures)


In [19]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — FIGURE 1: DEPTH HISTOGRAMS (2 × 3 grid)
# ═══════════════════════════════════════════════════════════════
from scipy.stats import gaussian_kde

LANGS_AVAIL = [l for l in LANGUAGES if l in results]
n_lang = len(LANGS_AVAIL)
ncols  = 3
nrows  = math.ceil(n_lang / ncols)

plt.style.use('dark_background')
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5*nrows))
fig.patch.set_facecolor('#0f0f1e')
axes = axes.flatten()

for i, lang in enumerate(LANGS_AVAIL):
    ax = axes[i]
    ax.set_facecolor('#1a1a2e')
    rd = results[lang]['real_depths'].astype(float)
    sd = results[lang]['rand_depths'].astype(float)

    bins = range(0, int(max(sd.max(), rd.max())) + 2)
    ax.hist(rd, bins=bins, density=True, alpha=0.55,
            color='#4e9af1', label='Real',   edgecolor='none')
    ax.hist(sd, bins=bins, density=True, alpha=0.45,
            color='#e94560', label='Random', edgecolor='none')

    # KDE
    for arr, col in [(rd,'#4e9af1'), (sd,'#e94560')]:
        xs  = np.linspace(arr.min(), arr.max(), 300)
        kde = gaussian_kde(arr, bw_method=0.35)
        ax.plot(xs, kde(xs), color=col, lw=2.0)

    ax.axvline(np.mean(rd), color='#4e9af1', ls='--', lw=1.5, alpha=0.9)
    ax.axvline(np.mean(sd), color='#e94560', ls='--', lw=1.5, alpha=0.9)

    h1  = stat_results[lang]['H1']
    p_s = f"p={h1['p']:.1e}" if h1['p'] > 0 else "p<10⁻³²³"
    ax.text(0.97, 0.93, p_s, transform=ax.transAxes,
            fontsize=8, color='#ccc', ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='#0f0f1e', alpha=0.7))

    ax.set_title(lang, color=LANG_COLORS[lang], fontsize=13, fontweight='bold')
    ax.set_xlabel("Tree Depth", fontsize=9, color='#aaa')
    ax.set_ylabel("Density",    fontsize=9, color='#aaa')
    ax.tick_params(colors='#888')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    if i == 0:
        ax.legend(fontsize=8, facecolor='#1a1a2e',
                  edgecolor='#444', labelcolor='white', loc='upper right')

# hide unused axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Figure 1 — Tree Depth Distribution: Real vs Random",
             fontsize=14, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig("fig1_depth_histograms.png", dpi=300,
            bbox_inches='tight', facecolor='#0f0f1e')
plt.show()
print("Figure 1 saved.")


Figure 1 saved.


In [20]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — FIGURE 2: MAX ARITY BOXPLOTS
# ═══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0f0f1e')
ax.set_facecolor('#1a1a2e')

x      = np.arange(len(LANGS_AVAIL))
width  = 0.32

bp_kw_real = dict(patch_artist=True, widths=width,
    boxprops    =dict(facecolor='#4e9af1', alpha=0.75, lw=1.2),
    medianprops =dict(color='white', lw=2.0),
    whiskerprops=dict(color='#4e9af1', lw=1.2),
    capprops    =dict(color='#4e9af1', lw=1.5),
    flierprops  =dict(marker='.', mfc='#4e9af1', ms=2, alpha=0.3))

bp_kw_rand = dict(patch_artist=True, widths=width,
    boxprops    =dict(facecolor='#e94560', alpha=0.75, lw=1.2),
    medianprops =dict(color='white', lw=2.0),
    whiskerprops=dict(color='#e94560', lw=1.2),
    capprops    =dict(color='#e94560', lw=1.5),
    flierprops  =dict(marker='.', mfc='#e94560', ms=2, alpha=0.3))

for i, lang in enumerate(LANGS_AVAIL):
    nsamp = min(3000, len(results[lang]['real_max_arity']))
    rng   = np.random.default_rng(SEED)
    ra    = rng.choice(results[lang]['real_max_arity'], nsamp, replace=False)
    sa    = rng.choice(results[lang]['rand_max_arity'], nsamp, replace=False)
    ax.boxplot([ra], positions=[x[i]-width/2], **bp_kw_real)
    ax.boxplot([sa], positions=[x[i]+width/2], **bp_kw_rand)

    # p-value annotation
    p = stat_results[lang]['H2']['p']
    p_s = "p<10⁻¹⁰" if p < 1e-10 else f"p={p:.3f}"
    ymax = ax.get_ylim()[1]
    ax.text(x[i], ymax*0.97, p_s, ha='center', va='top',
            fontsize=7.5, color='#aaa',
            bbox=dict(boxstyle='round', fc='#0f0f1e', alpha=0.6))

ax.set_xticks(x)
ax.set_xticklabels(LANGS_AVAIL, fontsize=11, color='white')
ax.set_ylabel("Max Node Arity", fontsize=11, color='#aaa')
ax.tick_params(colors='#888')
for sp in ax.spines.values(): sp.set_edgecolor('#333')
ax.grid(axis='y', color='#2a2a3e', lw=0.6)

handles = [mpatches.Patch(fc='#4e9af1', alpha=0.8, label='Real Trees'),
           mpatches.Patch(fc='#e94560', alpha=0.8, label='Random Trees')]
ax.legend(handles=handles, fontsize=10,
          facecolor='#0f0f1e', edgecolor='#444', labelcolor='white')
ax.set_title("Figure 2 — Max Node Arity: Real vs Random Dependency Trees",
             fontsize=13, fontweight='bold', color='white', pad=10)
plt.tight_layout()
plt.savefig("fig2_arity_boxplots.png", dpi=300,
            bbox_inches='tight', facecolor='#0f0f1e')
plt.show()
print("Figure 2 saved.")


Figure 2 saved.


In [21]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — FIGURE 3: CROSS-LINGUISTIC BAR CHART
# ═══════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0f0f1e')
x     = np.arange(len(LANGS_AVAIL))
w     = 0.35

for ax, rkey, skey, title, ylabel in [
    (ax1, 'real_depths',    'rand_depths',
         '(a) Mean Tree Depth',    'Mean Depth'),
    (ax2, 'real_max_arity', 'rand_max_arity',
         '(b) Mean Max Node Arity','Mean Max Arity'),
]:
    ax.set_facecolor('#1a1a2e')
    rm   = [np.mean(results[l][rkey]) for l in LANGS_AVAIL]
    sm   = [np.mean(results[l][skey]) for l in LANGS_AVAIL]
    rsem = [np.std(results[l][rkey], ddof=1)/np.sqrt(len(results[l][rkey]))
            for l in LANGS_AVAIL]
    ssem = [np.std(results[l][skey], ddof=1)/np.sqrt(len(results[l][skey]))
            for l in LANGS_AVAIL]

    b1 = ax.bar(x-w/2, rm, w, yerr=rsem, capsize=4,
                color='#4e9af1', alpha=0.85, label='Real',
                error_kw=dict(ecolor='white', elinewidth=1.2))
    b2 = ax.bar(x+w/2, sm, w, yerr=ssem, capsize=4,
                color='#e94560', alpha=0.85, label='Random',
                error_kw=dict(ecolor='white', elinewidth=1.2))

    for bar, lang in zip(b1, LANGS_AVAIL):
        bar.set_edgecolor(LANG_COLORS[lang]); bar.set_linewidth(2)

    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.05,
                f'{h:.1f}', ha='center', va='bottom',
                fontsize=7, color='#ccc')

    ax.set_xticks(x)
    ax.set_xticklabels(LANGS_AVAIL, rotation=20, ha='right',
                       fontsize=10, color='white')
    ax.set_ylabel(ylabel, fontsize=11, color='#aaa')
    ax.set_title(title, fontsize=12, color='white', pad=8)
    ax.tick_params(colors='#888')
    ax.grid(axis='y', color='#2a2a3e', lw=0.6)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.legend(fontsize=9, facecolor='#0f0f1e',
              edgecolor='#444', labelcolor='white')

fig.suptitle("Figure 3 — Cross-Linguistic Structural Comparison: Real vs Random Trees",
             fontsize=13, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig("fig3_cross_linguistic.png", dpi=300,
            bbox_inches='tight', facecolor='#0f0f1e')
plt.show()
print(" Figure 3 saved.")


 Figure 3 saved.


In [22]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — FIGURE 4: COHEN'S d HEATMAP
# ═══════════════════════════════════════════════════════════════
metrics_list = ['Depth (H1)', 'MaxArity (H2)']
hyp_keys     = ['H1', 'H2']
d_matrix     = np.zeros((len(LANGS_AVAIL), 2))

for i, lang in enumerate(LANGS_AVAIL):
    for j, hk in enumerate(hyp_keys):
        d_matrix[i, j] = stat_results[lang][hk]['d']

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor('#0f0f1e')
ax.set_facecolor('#1a1a2e')

im   = ax.imshow(d_matrix, cmap='RdBu', vmin=-2.5, vmax=2.5, aspect='auto')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("Cohen's d", color='#ccc', fontsize=10)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#ccc')

ax.set_xticks([0,1])
ax.set_yticks(range(len(LANGS_AVAIL)))
ax.set_xticklabels(metrics_list, fontsize=11, color='white')
ax.set_yticklabels(LANGS_AVAIL,  fontsize=11, color='white')

for i, lang in enumerate(LANGS_AVAIL):
    for j, hk in enumerate(hyp_keys):
        s   = stat_results[lang][hk]
        txt = f"{d_matrix[i,j]:+.2f}"
        if s['significant']:
            txt += "\n★"
        ax.text(j, i, txt, ha='center', va='center',
                color='white', fontsize=10, fontweight='bold')

ax.set_title("Figure 4 — Effect Size Heatmap (Cohen's d)\n"
             "★ = significant at Bonferroni α′",
             color='white', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig("fig4_cohens_d_heatmap.png", dpi=300,
            bbox_inches='tight', facecolor='#0f0f1e')
plt.show()
print(" Figure 4 saved.")


 Figure 4 saved.


In [23]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 — FIGURE 5: DEPTH vs SENTENCE LENGTH SCATTER
# Bonus figure: shows depth scales sublinearly with sentence length
# in real trees (linguistic constraint) but linearly in random trees
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.patch.set_facecolor('#0f0f1e')
axes = axes.flatten()

for i, lang in enumerate(LANGS_AVAIL):
    ax = axes[i]
    ax.set_facecolor('#1a1a2e')
    ns  = results[lang]['real_n']
    rd  = results[lang]['real_depths']
    sd  = results[lang]['rand_depths']

    # Bin by sentence length and compute mean depth per bin
    bins = np.arange(3, 81, 5)
    for arr, col, label in [(rd,'#4e9af1','Real'),(sd,'#e94560','Random')]:
        bin_means = []
        bin_mids  = []
        for b in range(len(bins)-1):
            mask = (ns >= bins[b]) & (ns < bins[b+1])
            if mask.sum() >= 5:
                bin_means.append(np.mean(arr[mask]))
                bin_mids.append((bins[b]+bins[b+1])/2)
        ax.plot(bin_mids, bin_means, 'o-', color=col, lw=2,
                ms=5, label=label, alpha=0.85)

    ax.set_title(lang, color=LANG_COLORS[lang],
                 fontsize=12, fontweight='bold')
    ax.set_xlabel("Sentence length (tokens)", fontsize=8, color='#aaa')
    ax.set_ylabel("Mean depth", fontsize=8, color='#aaa')
    ax.tick_params(colors='#888')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.grid(color='#2a2a3e', lw=0.5)
    if i == 0:
        ax.legend(fontsize=8, facecolor='#1a1a2e',
                  edgecolor='#444', labelcolor='white')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Figure 5 — Mean Depth vs Sentence Length: Real trees grow shallower than random",
    fontsize=13, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig("fig5_depth_vs_length.png", dpi=300,
            bbox_inches='tight', facecolor='#0f0f1e')
plt.show()
print(" Figure 5 saved.")


 Figure 5 saved.


In [24]:
# ═══════════════════════════════════════════════════════════════
# CELL 12 — FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
print(f"\nTotal sentences analysed: "
      f"{sum(len(results[l]['real_depths']) for l in LANGS_AVAIL):,}")
print(f"Languages: {', '.join(LANGS_AVAIL)}")
print(f"Bonferroni threshold: α′ = {ALPHA_ADJ:.5f}\n")

h1_total = sum(1 for l in LANGS_AVAIL
               if stat_results[l]['H1']['significant']
               and stat_results[l]['H1']['diff'] < 0)
h2_total = sum(1 for l in LANGS_AVAIL
               if stat_results[l]['H2']['significant']
               and stat_results[l]['H2']['diff'] > 0)

print(f"H1 (real trees SHALLOWER than random): {h1_total}/{len(LANGS_AVAIL)} languages ")
print(f"H2 (real trees HIGHER max arity than random): {h2_total}/{len(LANGS_AVAIL)} languages ")

print("\nPer-language summary:")
print(f"{'Language':<12} {'H1 Depth Δ':>12} {'H1 d':>8} {'H1 sig':>7} "
      f"{'H2 Arity Δ':>12} {'H2 d':>8} {'H2 sig':>7}")
print("-" * 70)
for lang in LANGS_AVAIL:
    h1 = stat_results[lang]['H1']
    h2 = stat_results[lang]['H2']
    print(f"{lang:<12} {h1['diff']:>+12.3f} {h1['d']:>+8.3f} "
          f"{'ok' if h1['significant'] else 'mismatch':>7} "
          f"{h2['diff']:>+12.3f} {h2['d']:>+8.3f} "
          f"{'ok' if h2['significant'] else 'mismatch':>7}")

print("\nOutput figures:")
for fig_name in ['fig1_depth_histograms.png','fig2_arity_boxplots.png',
                 'fig3_cross_linguistic.png','fig4_cohens_d_heatmap.png',
                 'fig5_depth_vs_length.png']:
    if os.path.exists(fig_name):
        kb = os.path.getsize(fig_name)//1024
        print(f"   {fig_name:<40} {kb} KB")
    else:
        print(f"   {fig_name} NOT FOUND")


FINAL RESULTS SUMMARY

Total sentences analysed: 65,623
Languages: English, Hindi, German, Japanese, Tamil, Sanskrit
Bonferroni threshold: α′ = 0.00417

H1 (real trees SHALLOWER than random): 6/6 languages 
H2 (real trees HIGHER max arity than random): 6/6 languages 

Per-language summary:
Language       H1 Depth Δ     H1 d  H1 sig   H2 Arity Δ     H2 d  H2 sig
----------------------------------------------------------------------
English            -3.279   -1.118      ok       +2.146   +1.547      ok
Hindi              -4.014   -1.611      ok       +2.781   +2.489      ok
German             -3.818   -1.520      ok       +2.333   +2.014      ok
Japanese           -4.351   -1.414      ok       +2.426   +1.743      ok
Tamil              -2.650   -1.031      ok       +2.107   +1.864      ok
Sanskrit           -0.960   -0.379      ok       +0.854   +0.752      ok

Output figures:
   fig1_depth_histograms.png                516 KB
   fig2_arity_boxplots.png                  132 KB
   fig3_